<a href="https://colab.research.google.com/github/tsilva/aiml-notebooks/blob/main/notebooks/wip-cartpole-dqn-lightning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

- TODO: compare dueling dqn vs dqn
- TODO: add noisy nets
- TODO: add multi-step dqn

# Solve Gymnasium with Rainbow

This notebook trains a **Deep Q-Network (DQN)** agent on the classic environment using **PyTorch Lightning**.

Install required libraries for Gymnasium, PyTorch Lightning, Tianshou, WandB, and notebook utilities.

In [ ]:
%pip install gymnasium[classic-control] pytorch-lightning tianshou wandb[media]>=0.20 tsilva_notebook_utils==0.0.92 > /dev/null

Load API keys and authentication tokens from Colab secrets for secure access.

🔑 Loading API keys and authentication tokens from Colab secrets:

In [ ]:
from tsilva_notebook_utils.colab import load_secrets_into_env

_ = load_secrets_into_env([
    'WANDB_API_KEY',
    'NOTIFICATION_URL',
    'NOTIFICATION_AUTH_TOKEN'
])

Define the configuration for the environment and agent hyperparameters.

In [ ]:
import os
import torch.nn as nn

def setup_config(env_id: str = "CartPole-v1") -> dict:
    # ------------- common hyper‑parameters (reasonable defaults) ----------------
    common = dict(
        env_id=env_id,
        seed=42,
        max_epochs=-1,                   # Unlimited by default; rely on reward target for early stop
        max_steps=-1,                    # Same as above but in terms of env steps
        log_freq=50,                     # Log every 50 env steps for fast feedback without spamming
        normalize_observation=True,      # Helps in continuous spaces; harmless for discrete ones
        replay_size=10_000,              # Plenty for small/state‑space tasks
        replay_alpha=0.6,                # Widely used compromise for PER
        replay_beta=0.4,                 # Start with partial bias‑correction (→1.0 over training if desired)
        target_update_freq=500           # 500 steps ≈ every 10 episodes on CartPole
    )

    # ------------- per‑environment specialist settings --------------------------
    env_configs = {
        # ------------------------------------------------------------------
        # CartPole‑v1: very small state‑space & dense reward.
        # Smaller network and fast epsilon anneal solve in <1 min on CPU.
        # ------------------------------------------------------------------
        "CartPole-v1": dict(
            hidden_sizes=(128, 128),      # Two 128‑unit layers hit >475 reward reliably
            gamma=0.997,                  # Slightly longer horizon helps balance pole
            batch_size=64,                # Larger batches smooth noisy updates
            min_buffer=1_000,             # One thousand transitions is enough to decorrelate
            replay_size=20_000,           # Modest but avoids memory bloat
            lr=5e-4,                      # Lower LR reduces overshoot in early training
            eps_start=1.0,                # Fully explore from scratch
            eps_end=0.05,                 # A touch more exploration near the end
            eps_total_steps=5_000,        # Anneal quickly – task is simple
            target_reward=475.0,          # OpenAI Gym ‘solved’ threshold
            target_update_freq=250        # More frequent to track fast changing value‑estimates
        ),

        # ------------------------------------------------------------------
        # MountainCar‑v0: sparse reward, needs long exploration.
        # Larger replay and slower ε decay are crucial.
        # ------------------------------------------------------------------
        "MountainCar-v0": dict(
            hidden_sizes=(256, 256),      # Extra capacity to approximate shaped potential
            gamma=0.99,                   # Standard discount – avoids diverging returns
            batch_size=64,                # Stable gradient estimates
            min_buffer=5_000,             # More experiences before learning to reduce bias
            replay_size=100_000,          # Large buffer captures rare successful episodes
            lr=2.5e-4,                    # Lower LR helps with sparse/unstable returns
            eps_start=1.0,                # Need heavy exploration
            eps_end=0.01,                 # Keep small exploratory tail forever
            eps_total_steps=50_000,       # Very slow anneal to continue searching
            target_reward=-110.0,         # Classic benchmark threshold
            target_update_freq=1_000      # Slow updates for stability
        )
    }

    if env_id not in env_configs:
        raise ValueError(f"Unsupported env_id: {env_id}")

    # ---------- merge and return (env‑specific overrides common) ---------------
    return {**common, **env_configs[env_id]}



CONFIG = setup_config()

Set the random seed for reproducibility.

In [ ]:
from tsilva_notebook_utils.lightning import seed_everything
seed_everything(CONFIG['seed'])

Login to Weights & Biases (wandb) for experiment tracking.

In [ ]:
from wandb import login
login()

Build and test the Gymnasium environment using the provided configuration.

In [ ]:
from tsilva_notebook_utils.misc import filter_kwargs
from tsilva_notebook_utils.gymnasium import build_env

build_env(**filter_kwargs(build_env, CONFIG))

Initialize the environment and determine the input and output dimensions for the model.

In [ ]:
env, _, _ = build_env(**filter_kwargs(build_env, CONFIG))
N_INPUTS = env.observation_space.shape[0]
N_OUTPUTS = int(env.action_space.n)
N_INPUTS, N_OUTPUTS

Create DQN model:

In [ ]:
class DQNModel(nn.Module):
    def __init__(self, n_inputs, hidden_sizes, n_outputs):
        super().__init__()

        layers = []
        input_size = n_inputs
        for hidden_size in hidden_sizes:
            layers.append(nn.Linear(input_size, hidden_size))
            layers.append(nn.ReLU())
            input_size = hidden_size
        layers.append(nn.Linear(input_size, n_outputs))
        self.net = nn.Sequential(*layers)

    def forward(self, x):
        return self.net(x)
    
model = DQNModel(N_INPUTS, CONFIG['hidden_sizes'], N_OUTPUTS)
model

Create Dueling DQN model:

In [ ]:
import torch.nn as nn
import torch

class DuelingDQNModel(nn.Module):
    """
    A feed-forward dueling network:
        • shared feature extractor
        • separate value (V) and advantage (A) streams
        • Q(s,a) = V(s) + A(s,a) − mean_a A(s,a)
    """
    def __init__(self, n_inputs, hidden_sizes, n_outputs):
        super().__init__()

        # --- shared feature layers ----------------------------------------
        layers = []
        last = n_inputs
        for h in hidden_sizes:
            layers += [nn.Linear(last, h), nn.ReLU()]
            last = h
        self.feature = nn.Sequential(*layers)

        # --- value stream --------------------------------------------------
        self.value = nn.Sequential(
            nn.Linear(last, last),
            nn.ReLU(),
            nn.Linear(last, 1),
        )

        # --- advantage stream ---------------------------------------------
        self.advantage = nn.Sequential(
            nn.Linear(last, last),
            nn.ReLU(),
            nn.Linear(last, n_outputs),
        )

    # ---------------------------------------------------------------------
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if not torch.is_tensor(x):                       # allow NumPy inputs
            x = torch.as_tensor(x, dtype=torch.float32)
        f = self.feature(x)
        v = self.value(f)                      # shape: (B, 1)
        a = self.advantage(f)                  # shape: (B, A)
        q = v + a - a.mean(dim=1, keepdim=True)
        return q

model = DuelingDQNModel(N_INPUTS, CONFIG['hidden_sizes'], N_OUTPUTS)
model

Implement the PyTorch Lightning module for DQN training, including replay buffer and training logic.

In [ ]:
import random
import numpy as np
import pytorch_lightning as pl
from tianshou.data import Batch, PrioritizedReplayBuffer
from tsilva_notebook_utils.torch import create_infinite_data_loader

infinite_data_loader = create_infinite_data_loader()

class DQNModule(pl.LightningModule):
    def __init__(self):
        super().__init__()

        self.save_hyperparameters()

        self.env_id = CONFIG['env_id']
        self.q_model = DQNModel(N_INPUTS, CONFIG['hidden_sizes'], N_OUTPUTS)
        self.target_model = DQNModel(N_INPUTS, CONFIG['hidden_sizes'], N_OUTPUTS)
        self.target_model.load_state_dict(self.q_model.state_dict())

        self.env, self.state, _ = build_env(**filter_kwargs(build_env, CONFIG))
        
        self.buffer = PrioritizedReplayBuffer(size=CONFIG['replay_size'], alpha=CONFIG['replay_alpha'], beta=CONFIG['replay_beta'])
        self.episode = 0
        self.total_steps = 0
        self.episode_steps = 0
        self.episode_reward = 0
        self.episode_shaped_reward = 0
        self.episode_rewards = []

    def forward(self, x):
        return self.q_model(x)

    @property
    def current_eps(self):
        eps_end = CONFIG['eps_end']
        eps_start = CONFIG['eps_start']
        eps_total_steps = CONFIG['eps_total_steps']
        eps = max(eps_end, eps_start - (eps_start - eps_end) * (self.total_steps / eps_total_steps))
        return eps
    
    def act(self, state):
        if random.random() < self.current_eps:
            return self.env.action_space.sample()
        else:
            state = torch.tensor(state, dtype=torch.float32, device=self.device).unsqueeze(0)
            with torch.no_grad(): q = self.q_model(state)
            return int(torch.argmax(q, dim=1)[0].item())

    def train_dataloader(self):
        return infinite_data_loader

    def training_step(self, batch, batch_idx):
        action = self.act(self.state)
        next_state, reward, terminated, truncated, info = self.env.step(action)
        shaped_reward = reward

        done = terminated or truncated
        self.buffer.add(Batch(
            obs=self.state,
            act=action,
            rew=shaped_reward,
            terminated=terminated,
            truncated=truncated,
            done=done,
            obs_next=next_state,
            info=info
        ))

        self.state = next_state

        self.total_steps += 1
        self.episode_steps += 1
        self.episode_reward += reward
        self.episode_shaped_reward += shaped_reward

        loss = None
        if len(self.buffer) >= CONFIG['min_buffer']:
            batch, indices = self.buffer.sample(CONFIG['batch_size'])
            states = torch.tensor(batch.obs, dtype=torch.float32, device=self.device)
            actions = torch.tensor(batch.act, dtype=torch.long, device=self.device).unsqueeze(-1)
            rewards = torch.tensor(batch.rew, dtype=torch.float32, device=self.device)
            next_states = torch.tensor(batch.obs_next, dtype=torch.float32, device=self.device)
            dones = torch.tensor(batch.done, dtype=torch.float32, device=self.device)
            weights = torch.tensor(batch.weight, dtype=torch.float32, device=self.device)
            
            #q_values = self.q_model(states).gather(1, actions).squeeze()
            #next_q = self.target_model(next_states).max(1)[0]
            #targets = rewards + CONFIG['gamma'] * next_q * (1 - dones)

            q_values = self.q_model(states).gather(1, actions).squeeze()

            # Double-DQN target: online net chooses the action, target net evaluates it
            next_online_actions = self.q_model(next_states).argmax(1, keepdim=True)
            next_q = self.target_model(next_states) \
                        .gather(1, next_online_actions) \
                        .squeeze()

            targets = rewards + CONFIG['gamma'] * next_q * (1 - dones)

            td_errors = (q_values - targets.detach()).abs()
            priorities = (td_errors + 1e-6) # TODO: softcode this
            self.buffer.update_weight(indices, priorities)

            # Optionally use importance-sampling weights for loss
            loss = (weights * nn.functional.mse_loss(q_values, targets.detach(), reduction='none')).mean()
            self.log('loss', loss, on_step=True, prog_bar=True)

            if self.global_step > 0 and self.global_step % CONFIG['target_update_freq'] == 0:
                self.target_model.load_state_dict(self.q_model.state_dict())
                
        if done:
            self.episode_rewards.append(self.episode_reward)
            
            # Compute stats
            rewards_arr = np.array(self.episode_rewards[-100:])  # Last 100 episodes
            min_r = float(np.min(rewards_arr))
            max_r = float(np.max(rewards_arr))
            mean_r = float(np.mean(rewards_arr))
            std_r = float(np.std(rewards_arr))

            # Log stats
            self.log('episode', self.episode, on_step=True, prog_bar=True)
            self.log('reward', self.episode_reward, on_step=True, prog_bar=True)
            self.log('shaped_reward', self.episode_shaped_reward, on_step=True, prog_bar=True)
            self.log('steps', self.episode_steps, on_step=True, prog_bar=True)
            self.log('eps', self.current_eps, on_step=True, prog_bar=True)
            self.log('reward_min', min_r, on_step=True, prog_bar=True)
            self.log('reward_max', max_r, on_step=True, prog_bar=True)
            self.log('reward_mean', mean_r, on_step=True, prog_bar=True)
            self.log('reward_std', std_r, on_step=True, prog_bar=True)

            self.episode_steps = 0
            self.episode_reward = 0
            self.episode_shaped_reward = 0
            self.episode += 1
            self.state = self.env.reset()[0]

        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.q_model.parameters(), lr=CONFIG['lr'])

module = DQNModule()
module

Set up the PyTorch Lightning trainer and start training the DQN agent.

In [ ]:
import os
import pytorch_lightning as pl
from pytorch_lightning.loggers import WandbLogger

from tsilva_notebook_utils.lightning import StopOnLambda
from tsilva_notebook_utils.gymnasium import build_pl_callback

trainer = pl.Trainer(
    max_epochs=CONFIG['max_epochs'],
    max_steps=CONFIG['max_steps'],
    log_every_n_steps=CONFIG['log_freq'],
    logger=WandbLogger(project=os.getenv('NOTEBOOK_ID'), config=CONFIG),
    enable_model_summary=False,
    callbacks=[
        StopOnLambda(
            lambda metrics: metrics.get('reward_mean', -float('inf')) >= CONFIG['target_reward'],
            message=f"Stopping: reward_mean >= {CONFIG['target_reward']}"
        ),
        build_pl_callback("EvalEpisodeAndRecordCallback", every_n_episodes=10)
    ]
)
trainer.fit(module)

Render and visualize a trained episode using the learned Q-network.

In [ ]:
from tsilva_notebook_utils.gymnasium import render_episode

render_episode(
    env_id=CONFIG["env_id"],
    model=module.q_model
)